# **Which customers are leaving, why, and who should an account manager contact first?**

imports and loading

In [9]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix

pd.set_option("display.max_colwidth", None)

data = pd.read_csv("/content/WA_Fn-UseC_-Telco-Customer-Churn.csv")
print(data.shape)
data.head()

(7043, 21)


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


Cleaning

In [10]:
ids = data["customerID"]  # kept for the call list
data = data.drop(columns=["customerID", "gender"])

data["TotalCharges"] = pd.to_numeric(data["TotalCharges"], errors="coerce").fillna(0)

print("-----------------------------------------")
print(data["TotalCharges"].dtype)
print("-----------------------------------------")

# transforming Churn column values into 1/0:
data["Churn"] = data["Churn"].map({"Yes": 1, "No": 0})
print(data["Churn"].head())

-----------------------------------------
float64
-----------------------------------------
0    0
1    0
2    1
3    0
4    1
Name: Churn, dtype: int64


encode, split, scale, train

In [11]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix

# 1) features and target
X = pd.get_dummies(data.drop(columns="Churn"), drop_first=True, dtype=int)
y = data["Churn"]

# 2) split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 3) scale
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

# 4) train
model = LogisticRegression(max_iter=1000, class_weight="balanced")
model.fit(X_train_s, y_train)

LogisticRegression(class_weight='balanced', max_iter=1000)

evaluate

In [12]:
pred = model.predict(X_test_s)
print(classification_report(y_test, pred))
print("------------------------------------------")
print(confusion_matrix(y_test, pred))

              precision    recall  f1-score   support

           0       0.90      0.72      0.80      1035
           1       0.51      0.78      0.61       374

    accuracy                           0.74      1409
   macro avg       0.70      0.75      0.71      1409
weighted avg       0.80      0.74      0.75      1409

------------------------------------------
[[748 287]
 [ 81 293]]


What drives churn

Tenure and longer contracts push toward staying. Fiber optic, streaming services, and electronic check push toward leaving.

Note: `MonthlyCharges` and `TotalCharges` overlap (total is roughly tenure x monthly), so their individual coefficients shouldn't be interpreted on their own.



---



coefficients

In [13]:
coefs = pd.Series(model.coef_[0], index=X.columns).sort_values()
print(coefs.head(8))   # push toward staying
print("-----------------------------------------")
print(coefs.tail(8))   # push toward leaving

tenure                             -1.162318
MonthlyCharges                     -1.034729
Contract_Two year                  -0.616906
Contract_One year                  -0.296124
OnlineSecurity_Yes                 -0.115971
Dependents_Yes                     -0.104325
InternetService_No                 -0.098945
OnlineBackup_No internet service   -0.098945
dtype: float64
-----------------------------------------
DeviceProtection_Yes              0.057881
PaperlessBilling_Yes              0.164167
PaymentMethod_Electronic check    0.188373
MultipleLines_Yes                 0.200685
StreamingTV_Yes                   0.272050
StreamingMovies_Yes               0.284380
TotalCharges                      0.499528
InternetService_Fiber optic       0.805734
dtype: float64




---



# Risk scores for current customers

Every customer gets a churn risk score. Customers who already churned are removed, since they can't be retained. The scores are a ranking, not literal probabilities, because of the balanced class weights.



---



scores and call list table

In [14]:
# Churn probability for every customer
risk = model.predict_proba(scaler.transform(X))[:, 1]

calls = pd.DataFrame({
    "customerID": ids,
    "churn_risk": risk.round(3),
    "MonthlyCharges": data["MonthlyCharges"],
    "tenure": data["tenure"],
    "Contract": data["Contract"],
    "InternetService": data["InternetService"],
    "TechSupport": data["TechSupport"],
    "PaymentMethod": data["PaymentMethod"],
    "AlreadyChurned": data["Churn"],
})

# Keep only current customers
calls = calls[calls["AlreadyChurned"] == 0].drop(columns="AlreadyChurned")
print(len(calls), "current customers")

5174 current customers




---



Tiers and suggested actions

- **Tier 1, call this week:** risk of 0.85 or more and a bill at or above the median (personal call from an account manager)
- **Tier 2, check in this month:** risk of 0.5 or more (automated email or SMS with an offer)
- **Tier 3, monitor:** everyone else

The threshold of 0.85 gives a list a team could work through in a week.

Each customer also gets suggested actions that map to the dashboard findings. The primary action is the first matching rule.



---



tiers and actions

In [15]:
median_bill = calls["MonthlyCharges"].median()



def tier(row):
    if row["churn_risk"] >= 0.85 and row["MonthlyCharges"] >= median_bill:
        return "1 - Call this week"
    if row["churn_risk"] >= 0.5:
        return "2 - Check in this month"
    return "3 - Monitor"

calls["Tier"] = calls.apply(tier, axis=1)

# Suggested actions for each customer
def action(row):
    acts = []
    if row["Contract"] == "Month-to-month":
        acts.append("Offer longer-contract discount")
    if row["PaymentMethod"] == "Electronic check":
        acts.append("Offer auto-pay credit")
    if row["InternetService"] == "Fiber optic" and row["TechSupport"] == "No":
        acts.append("Offer tech support trial")
    if row["tenure"] <= 10:
        acts.append("Onboarding check-in")
    return acts

all_actions = calls.apply(action, axis=1)
calls["Primary action"] = all_actions.apply(lambda a: a[0] if a else "General check-in")
calls["Suggested actions"] = all_actions.apply(lambda a: "; ".join(a) if a else "General check-in")

calls = calls.sort_values(["Tier", "churn_risk"], ascending=[True, False])

print("---------------------------------------------------")

print(calls["Tier"].value_counts())
print(calls.head(10))

---------------------------------------------------
Tier
3 - Monitor                3777
2 - Check in this month    1261
1 - Call this week          136
Name: count, dtype: int64
      customerID  churn_risk  MonthlyCharges  tenure        Contract  \
3159  5150-ITWWB       0.933           94.85       3  Month-to-month   
3346  2545-EBUPK       0.922           84.05       2  Month-to-month   
935   6630-UJZMY       0.918           83.25       4  Month-to-month   
4039  8161-QYMTT       0.917           94.10       7  Month-to-month   
5213  7668-XCFYV       0.913           92.55      17  Month-to-month   
293   4847-TAJYI       0.912           89.35       6  Month-to-month   
4150  0187-QSXOE       0.911           89.00       7  Month-to-month   
4618  6350-XFYGW       0.911           94.75       4  Month-to-month   
345   0021-IKXGC       0.910           72.10       1  Month-to-month   
5140  7577-SWIFR       0.910           89.25       1  Month-to-month   

     InternetService TechSup

## 7. Revenue by tier and export

Monthly revenue sitting in each tier, and the call list exported to Excel.



---



revenue and export

In [16]:
print(calls.groupby("Tier")["MonthlyCharges"].agg(["count", "sum", "mean"]).round(0))

calls.to_excel("churn_call_list.xlsx", index=False)

                         count       sum  mean
Tier                                          
1 - Call this week         136   11657.0  86.0
2 - Check in this month   1261   93624.0  74.0
3 - Monitor               3777  211704.0  56.0
